In [ ]:
!unzip -oq open_data.zip && unzip -oq open.zip -d open && unzip -oq baseline_submit.zip -d base
import pandas as pd, numpy as np, sys
sys.path.insert(0, 'base/model')  # 베이스라인 함수 재사용
train = pd.read_csv('open/train.csv')
cat = pd.read_csv('open/vessel_catalog.csv')
train['is_kor'] = train.vessel_name.str.contains('[가-힣]')  # 평가 기준: 선박명 한글 여부
print(train.shape, cat.shape)
print(train.is_kor.value_counts())

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, va_idx = next(gss.split(train, groups=train.target_vessel_id))
tr, va = train.iloc[tr_idx].reset_index(drop=True), train.iloc[va_idx].reset_index(drop=True)
print(len(tr), len(va), '| 선박 겹침:', len(set(tr.target_vessel_id) & set(va.target_vessel_id)))
print('검증 한글 행:', va.is_kor.sum(), '/ 비한글:', (~va.is_kor).sum())

In [ ]:
import joblib
from sklearn.base import clone
from baseline_lib import CatalogIndex, make_training_pairs, predict_ids, local_weighted_accuracy
idx = CatalogIndex(cat)
X, y, rec_tr = make_training_pairs(tr, idx)          # 정답 1 + 후보 오답 최대 20
rf = clone(joblib.load('base/model/rf.pkl')['model']).fit(X, y)
va['pred'] = predict_ids(rf, va, idx)
va['hit'] = [idx.id_to_index[t] in idx.candidates(r) for r, t in zip(va.to_dict('records'), va.target_vessel_id)]
va['ok'] = va.pred == va.target_vessel_id
print(local_weighted_accuracy(va, va.pred.tolist()))
print(va.groupby('is_kor')[['hit', 'ok']].mean().round(4))  # hit=후보 재현율, ok=정확도

In [ ]:
m = va[va.is_kor].merge(cat[['vessel_id','vessel_name','gross_tonnage']], left_on='target_vessel_id', right_on='vessel_id', suffixes=('','_cat'))
m['gt_gap'] = (m.gross_tonnage - m.gross_tonnage_cat).abs() / m.gross_tonnage_cat  # 상대 톤수 오차
print(m[['vessel_name','vessel_name_cat','gross_tonnage','gross_tonnage_cat','ok']].sample(25, random_state=1).to_string())
print(m.gt_gap.describe().round(3))
print('톤수 결측:', m.gross_tonnage.isna().mean().round(3), '| 오차 0:', (m.gt_gap==0).mean().round(3))

In [ ]:
%%writefile phon.py
import re, unicodedata
CHO = ['K','K','N','T','T','R','M','P','P','S','S','','C','C','C','K','T','P','H']
JONG = ['','K','K','K','N','N','N','T','R','R','R','R','R','R','R','R','M','P','P','T','T','NG','T','T','K','T','P','T']
LET = dict(A='',B='P',C='S',D='T',E='',F='P',G='C',H='C',I='',J='C',K='K',L='R',M='M',N='N',O='',P='P',Q='K',R='R',S='S',T='T',U='',V='P',W='TPR',X='KS',Y='',Z='C')
V = set('AEIOUY')

def kor_key(s):  # 한글 -> 자음 골격
    s = unicodedata.normalize('NFKC', str(s))
    s = re.sub(r'제\s*(\d+)', r'\1', s); s = re.sub(r'호$', '', s.strip())  # 제2..호 -> 2
    out = []
    for ch in s:
        o = ord(ch) - 0xAC00
        if 0 <= o < 11172: out.append(CHO[o // 588] + JONG[o % 28])  # 초성+종성, 모음 버림
        elif ch.isdigit(): out.append(ch)
    return re.sub(r'(.)\1+', r'\1', ''.join(out))

def eng_word(w):
    if w.isdigit(): return w
    if len(w) <= 3 and not any(c in V for c in w):  # 약어 SJ -> 에스제이
        return ''.join(LET.get(c, '') for c in w)
    w = w.replace('PH','F').replace('CK','K').replace('QU','K').replace('SH','S').replace('TH','S').replace('X','KS')
    out = []
    for i, c in enumerate(w):
        nx = w[i+1] if i+1 < len(w) else ''; pv = w[i-1] if i > 0 else ''
        if c in V or c == 'W': continue
        if c == 'H' and pv and pv not in V and pv != 'C': continue
        if c == 'R' and pv in V and nx not in V: continue  # 무음 r: Morning -> 모닝
        if c == 'C': out.append('C' if nx == 'H' else ('S' if nx and nx in 'EIY' else 'K')); continue
        if c == 'H' and pv == 'C': continue
        if c == 'N' and nx == 'G': out.append('NG'); continue
        if c == 'G' and pv == 'N': continue
        out.append({'G':'K','J':'C','Z':'C','V':'P','F':'P','L':'R','D':'T','B':'P'}.get(c, c))
    return ''.join(out)

def eng_key(s):  # 영문 -> 자음 골격
    s = unicodedata.normalize('NFKC', str(s)).upper()
    s = re.sub(r'\bNO\.?\s*(?=\d)', '', s)  # No.2 -> 2
    return re.sub(r'(.)\1+', r'\1', ''.join(eng_word(t) for t in re.findall(r'[A-Z]+|\d+', s)))

In [ ]:
!pip -q install rapidfuzz
from rapidfuzz import fuzz, process
from phon import kor_key, eng_key
cat['key'] = cat.vessel_name.map(eng_key)
gt, keys, ids = cat.gross_tonnage.values, cat.key.values, cat.vessel_id.values

def pred_kor(name, g, a=30):
    m = np.where(np.abs(gt - g) / np.maximum(gt, 1) <= 0.35)[0]  # 톤수 ±35% 후보
    s = process.cdist([kor_key(name)], list(keys[m]), scorer=fuzz.ratio)[0] - a * np.abs(gt[m] - g) / np.maximum(gt[m], 1)
    return ids[m[np.argmax(s)]]

vk = va[va.is_kor].copy()
vk['pred2'] = [pred_kor(n, g) for n, g in zip(vk.vessel_name, vk.gross_tonnage)]
print('한글 정확도: 베이스라인', vk.ok.mean().round(4), '→ 골격', (vk.pred2 == vk.target_vessel_id).mean().round(4))

In [ ]:
id2name = dict(zip(cat.vessel_id, cat.vessel_name)); id2gt = dict(zip(cat.vessel_id, cat.gross_tonnage)); id2key = dict(zip(cat.vessel_id, cat.key))
e = vk[vk.pred2 != vk.target_vessel_id].copy()
e['q_key'] = e.vessel_name.map(kor_key)
e['true'] = e.target_vessel_id.map(id2name); e['true_key'] = e.target_vessel_id.map(id2key); e['true_gt'] = e.target_vessel_id.map(id2gt)
e['pred'] = e.pred2.map(id2name); e['pred_key'] = e.pred2.map(id2key); e['pred_gt'] = e.pred2.map(id2gt)
print(len(e), '개 오답')
print(e[['vessel_name','q_key','gross_tonnage','true','true_key','true_gt','pred','pred_key','pred_gt']].to_string())

In [ ]:
from google.colab import files
up = files.upload()  # byt5_pairs.csv, byt5_cands.csv 선택
!ls -la *.csv
!nvidia-smi --query-gpu=name,memory.total --format=csv  # L4인지 확인


In [ ]:
import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
P = pd.read_csv('byt5_pairs.csv'); D = pd.read_csv('byt5_cands.csv')
rng = np.random.RandomState(0); vs = P.vessel.unique(); rng.shuffle(vs); hold = set(vs[:len(vs)//5])  # 검증 선박
tr = P[~P.vessel.isin(hold)]
pairs = list(zip(tr.eng, tr.kor))
for k, e in zip(tr.kor, tr.eng):  # 단어 수 같으면 단어쌍도 추가 (데이터 늘리기)
    kw, ew = k.split(), e.replace('-', ' ').split()
    if len(kw) == len(ew) > 1: pairs += list(zip(ew, kw))
print('학습쌍', len(pairs))
tok = AutoTokenizer.from_pretrained('google/byt5-small')
model = T5ForConditionalGeneration.from_pretrained('google/byt5-small').cuda()
opt = torch.optim.AdamW(model.parameters(), lr=5e-4); model.train()
for ep in range(30):
    rng.shuffle(pairs)
    for i in range(0, len(pairs), 32):
        b = pairs[i:i+32]
        x = tok([e for e, _ in b], return_tensors='pt', padding=True).to('cuda')
        y = tok([k for _, k in b], return_tensors='pt', padding=True).input_ids.to('cuda'); y[y == tok.pad_token_id] = -100
        loss = model(**x, labels=y).loss; loss.backward(); opt.step(); opt.zero_grad()
    if ep % 5 == 4: print(ep, round(loss.item(), 4))

In [ ]:
model.eval()
@torch.no_grad()
def ll(eng, kor, bs=512):  # log P(한글 | 영문)
    out = []
    for i in range(0, len(eng), bs):
        x = tok(eng[i:i+bs], return_tensors='pt', padding=True).to('cuda')
        y = tok(kor[i:i+bs], return_tensors='pt', padding=True).input_ids.to('cuda'); m = y != tok.pad_token_id
        lp = torch.log_softmax(model(**x, labels=y.masked_fill(~m, -100)).logits.float(), -1)
        out += (lp.gather(-1, y.unsqueeze(-1)).squeeze(-1) * m).sum(1).tolist()
    return np.array(out)
H = D[D.vessel.isin(hold)].copy(); H['ll'] = ll(H.cand_name.tolist(), H.kor.tolist())
print('검증 행', H.qid.nunique(), '| ByT5 점수만으로 정확도', H.loc[H.groupby('qid').ll.idxmax()].y.mean().round(4))
s = P[P.vessel.isin(hold)].head(12)
g = model.generate(**tok(s.eng.tolist(), return_tensors='pt', padding=True).to('cuda'), max_length=64)
print(pd.DataFrame({'eng': s.eng, '정답': s.kor, '생성': tok.batch_decode(g, skip_special_tokens=True)}).to_string())

In [ ]:
from google.colab import files
up = files.upload()  # byt5_pairs.csv(새 버전), byt5_all_cands.csv

In [ ]:
import os, glob
print(glob.glob('byt5*'))
for f in glob.glob('byt5_pairs (*).csv'): os.replace(f, 'byt5_pairs.csv')
for f in glob.glob('byt5_all_cands (*).csv'): os.replace(f, 'byt5_all_cands.csv')
print(pd.read_csv('byt5_pairs.csv').columns.tolist())  # fold 있으면 OK

In [ ]:
import time
P = pd.read_csv('byt5_pairs.csv'); E = pd.read_csv('byt5_all_cands.csv')

def train(df, epochs=30, seed=0):
    r = np.random.RandomState(seed); torch.manual_seed(seed)
    pr = list(zip(df.eng, df.kor))
    for k, e in zip(df.kor, df.eng):
        kw, ew = k.split(), e.replace('-', ' ').split()
        if len(kw) == len(ew) > 1: pr += list(zip(ew, kw))
    m = T5ForConditionalGeneration.from_pretrained('google/byt5-small').cuda()
    o = torch.optim.AdamW(m.parameters(), lr=5e-4); m.train()
    for ep in range(epochs):
        r.shuffle(pr)
        for i in range(0, len(pr), 32):
            b = pr[i:i+32]
            x = tok([e for e, _ in b], return_tensors='pt', padding=True).to('cuda')
            y = tok([k for _, k in b], return_tensors='pt', padding=True).input_ids.to('cuda'); y[y == tok.pad_token_id] = -100
            m(**x, labels=y).loss.backward(); o.step(); o.zero_grad()
    return m.eval()

E['ll'] = np.nan
for f in range(5):
    t0 = time.time(); model = train(P[P.fold != f])  # ll() 함수가 전역 model을 씀
    ix = E.index[E.fold == f]; E.loc[ix, 'll'] = ll(E.cand_name[ix].tolist(), E.kor[ix].tolist())
    print('fold', f, len(ix), round(time.time() - t0), 's')
E[['ll']].to_csv('byt5_ll.csv', index=False)

model = train(P)  # 최종 모델 (제출용)
model.half().save_pretrained('byt5_final', safe_serialization=True); tok.save_pretrained('byt5_final')
!du -sh byt5_final
files.download('byt5_ll.csv')

In [ ]:
import os, glob, shutil
up = files.upload()  # submit_v5_code.zip, open_data.zip
for f in glob.glob('* (*).zip'): os.replace(f, f.split(' (')[0] + '.zip')
!rm -rf sub && mkdir sub && unzip -oq submit_v5_code.zip -d sub
!unzip -oq open_data.zip && unzip -oq open.zip -d open
shutil.copytree('byt5_final', 'sub/model/byt5_final')
!ls sub sub/model; du -sh sub/model/byt5_final

In [ ]:
!pip -q install rapidfuzz lightgbm
!rm -rf /content/tf457 && pip -q install --no-deps --target /content/tf457 transformers==4.57.6 huggingface-hub==0.36.2 tokenizers==0.22.1
!PYTHONPATH=/content/tf457 python -c "import transformers, numpy; print(transformers.__version__, numpy.__version__)"
!mkdir -p sub/data && cp open/vessel_catalog.csv sub/data/
t = pd.read_csv('open/train.csv'); s = t.sample(400, random_state=1)
s.drop(columns='target_vessel_id').to_csv('sub/data/test.csv', index=False)
!cd sub && time PYTHONPATH=/content/tf457 python script.py 2>&1 | grep -v -i warn | tail -3
o = pd.read_csv('sub/output/submission.csv').merge(s[['sample_id','vessel_name','target_vessel_id']], on='sample_id', suffixes=('','_t'))
k = o.vessel_name.str.contains('[가-힣]'); ok = o.target_vessel_id == o.target_vessel_id_t
print('한글', ok[k].mean().round(3), '비한글', ok[~k].mean().round(3))

In [ ]:
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
!rm -rf sub/data sub/output && cd sub && zip -r -q ../submit_v5.zip model script.py requirements.txt -x '*__pycache__*'
!ls -la submit_v5.zip
files.download('submit_v5.zip')

In [ ]:
import os; print(os.path.exists('byt5_final'))

In [ ]:
!pip -q install rapidfuzz lightgbm
import pandas as pd, numpy as np, torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from google.colab import files
up = files.upload()  # byt5_pairs.csv
tok = AutoTokenizer.from_pretrained('google/byt5-small')
P = pd.read_csv('byt5_pairs.csv'); r = np.random.RandomState(0); torch.manual_seed(0)
pr = list(zip(P.eng, P.kor))
for k, e in zip(P.kor, P.eng):
    kw, ew = k.split(), e.replace('-', ' ').split()
    if len(kw) == len(ew) > 1: pr += list(zip(ew, kw))
model = T5ForConditionalGeneration.from_pretrained('google/byt5-small').cuda()
o = torch.optim.AdamW(model.parameters(), lr=5e-4); model.train()
for ep in range(30):
    r.shuffle(pr)
    for i in range(0, len(pr), 32):
        b = pr[i:i+32]
        x = tok([e for e, _ in b], return_tensors='pt', padding=True).to('cuda')
        y = tok([k for _, k in b], return_tensors='pt', padding=True).input_ids.to('cuda'); y[y == tok.pad_token_id] = -100
        model(**x, labels=y).loss.backward(); o.step(); o.zero_grad()
model.eval().half().save_pretrained('byt5_final', safe_serialization=True); tok.save_pretrained('byt5_final')